# 🏥 CareFlow — Notebook 6: DBT SQL Models Ported to Python & Pandas
## Modern Data Engineering & Transformation Pipeline

**Author:** CareFlow Analytics Team  
**Dataset:** Emergency Department Event Logs (`simulator/output/ehr_event_log.csv`)  
**Scope:** 1:1 Pure Python replacement for DBT SQL transformations (`stg_ehr_events`, `fct_patient_journeys`, `mart_bottlenecks`, `mart_resource_handovers`, `mart_sla_compliance`).

---

### Key Data Engineering Architecture:
1. **Staging (`stg_ehr_events`):** Standardizes types, extracts peak morning rush hours (8-11 AM), and computes chronological lag window functions (`shift(1)`).
2. **Fact Table (`fct_patient_journeys`):** Aggregates event sequences to case level, calculates Total Length of Stay (LOS), identifies primary attending physician, and flags loopback & triage bypass anomalies.
3. **Marts:**
   - `mart_bottlenecks`: Inter-activity transition queue delays and peak vs off-peak wait times.
   - `mart_resource_handovers`: Inter-role latency and communication friction rating (`Critical Friction`, `Moderate Friction`, `Nominal Velocity`).
   - `mart_sla_compliance`: Door-to-Doctor, Triage-to-XRay, and CMS OP-18 ED boarding breach tracking.


In [ ]:
# 1. Imports and Configuration
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('dark_background')
sns.set_palette('crest')

CSV_PATH = '../simulator/output/ehr_event_log.csv'
OUTPUT_DIR = '../careflow_analytics/output'
os.makedirs(OUTPUT_DIR, exist_ok=True)

raw_df = pd.read_csv(CSV_PATH)
print(f"Loaded raw EHR log: {len(raw_df):,} events across {raw_df['Case_ID'].nunique():,} cases.")
raw_df.head()


### 2. Staging Layer: `stg_ehr_events`
**DBT SQL Equivalent:**
```sql
select
    case_id, activity_name, event_timestamp, resource_name,
    patient_gender, patient_age_group, clinical_severity,
    extract(hour from event_timestamp) as event_hour,
    case when extract(hour from event_timestamp) between 8 and 11 then true else false end as is_morning_rush,
    lag(activity_name) over (partition by case_id order by event_timestamp) as previous_activity,
    lag(event_timestamp) over (partition by case_id order by event_timestamp) as previous_event_timestamp
from raw_source;
```


In [ ]:
def run_stg_ehr_events(df_in: pd.DataFrame) -> pd.DataFrame:
    df = df_in.copy()
    df["case_id"] = df["Case_ID"].astype(str)
    df["activity_name"] = df["Activity_Name"].astype(str)
    df["event_timestamp"] = pd.to_datetime(df["Timestamp"])
    df["resource_name"] = df["Resource"].astype(str)
    df["patient_gender"] = df["Gender"].astype(str)
    df["patient_age_group"] = df["Age_Group"].astype(str)
    df["clinical_severity"] = df["Severity"].astype(str)

    df = df.sort_values(by=["case_id", "event_timestamp"]).reset_index(drop=True)
    df["event_hour"] = df["event_timestamp"].dt.hour
    df["is_morning_rush"] = df["event_hour"].between(8, 11)

    # Window lag functions
    df["previous_activity"] = df.groupby("case_id")["activity_name"].shift(1)
    df["previous_event_timestamp"] = df.groupby("case_id")["event_timestamp"].shift(1)
    return df

stg_df = run_stg_ehr_events(raw_df)
print(f"Staging materialized: {len(stg_df):,} rows.")
stg_df[['case_id', 'activity_name', 'event_timestamp', 'is_morning_rush', 'previous_activity']].head(6)


### 3. Fact Table: `fct_patient_journeys`
Aggregates event stream to case level, calculating throughput duration and domain-specific process mining anomalies:
- **Loopback Rework:** Patients with X-Ray that require repeated Triage ($>1$).
- **Bypass Anomaly:** High severity patients bypassing formal triage registration.


In [ ]:
def run_fct_patient_journeys(stg: pd.DataFrame) -> pd.DataFrame:
    grouped = stg.groupby("case_id")
    fct = grouped.agg(
        patient_gender=("patient_gender", "first"),
        patient_age_group=("patient_age_group", "first"),
        clinical_severity=("clinical_severity", "first"),
        visit_start_time=("event_timestamp", "min"),
        visit_end_time=("event_timestamp", "max"),
        total_events_count=("activity_name", "count"),
        triage_count=("activity_name", lambda s: (s == "Triage").sum()),
        xray_count=("activity_name", lambda s: (s == "X-Ray").sum()),
    ).reset_index()

    # Assign primary physician
    doc_events = stg[stg["activity_name"].isin(["Doctor Consultation", "Treatment"])]
    doc_map = doc_events.groupby("case_id")["resource_name"].last()
    fct["primary_physician"] = fct["case_id"].map(doc_map).fillna("Unassigned / Nurse")

    fct["total_los_minutes"] = ((fct["visit_end_time"] - fct["visit_start_time"]).dt.total_seconds() / 60.0).round(1)
    fct["has_radiology_loopback_rework"] = (fct["xray_count"] > 0) & (fct["triage_count"] > 1)
    fct["is_triage_bypassed_anomaly"] = fct["triage_count"] == 0
    return fct

fct_df = run_fct_patient_journeys(stg_df)
print(f"Fact table materialized: {len(fct_df):,} patient journeys.")
print(f"Loopback cases: {fct_df['has_radiology_loopback_rework'].sum():,} ({fct_df['has_radiology_loopback_rework'].mean()*100:.1f}%)")
print(f"Triage bypassed: {fct_df['is_triage_bypassed_anomaly'].sum():,} ({fct_df['is_triage_bypassed_anomaly'].mean()*100:.1f}%)")
fct_df.head()


### 4. Operational Marts: Bottlenecks & Handovers
Calculates queue transitions, wait distributions, and inter-role communication friction ratings.


In [ ]:
# Mart: Bottlenecks
trans = stg_df[stg_df["previous_activity"].notna()].copy()
trans["wait_min"] = (trans["event_timestamp"] - trans["previous_event_timestamp"]).dt.total_seconds() / 60.0

mart_bottlenecks = trans.groupby(["previous_activity", "activity_name", "is_morning_rush"]).agg(
    volume=("wait_min", "count"),
    avg_wait=("wait_min", lambda s: round(s.mean(), 1)),
    median_wait=("wait_min", lambda s: round(s.median(), 1)),
    max_wait=("wait_min", lambda s: round(s.max(), 1))
).reset_index().rename(columns={"previous_activity": "source", "activity_name": "target"})

print("Top 5 Bottlenecks by Volume:")
mart_bottlenecks.sort_values(by="volume", ascending=False).head()


In [ ]:
# Mart: Resource Handovers
stg_df["source_res"] = stg_df.groupby("case_id")["resource_name"].shift(1).fillna("Triage Staff")
stg_df["target_res"] = stg_df["resource_name"]

handovers = stg_df[stg_df["previous_activity"].notna()].copy()
handovers["latency_min"] = (handovers["event_timestamp"] - handovers["previous_event_timestamp"]).dt.total_seconds() / 60.0

mart_handovers = handovers.groupby(["source_res", "target_res"]).agg(
    volume=("latency_min", "count"),
    avg_latency=("latency_min", lambda s: round(s.mean(), 1)),
    median_latency=("latency_min", lambda s: round(s.median(), 1))
).reset_index()

def rate_friction(val):
    if val > 40: return "Critical Friction"
    elif val > 25: return "Moderate Friction"
    return "Nominal Velocity"

mart_handovers["friction_rating"] = mart_handovers["avg_latency_min"].apply(rate_friction)
print("Top 5 Inter-Departmental Handovers:")
mart_handovers.sort_values(by="volume", ascending=False).head()


### 5. Mart: SLA Compliance & Quality Benchmarking
Evaluates clinical timing thresholds:
- Door-to-Doctor ($<30$ minutes)
- Triage-to-X-Ray ($<45$ minutes)
- Total Length of Stay CMS OP-18 ($<240$ minutes)


In [ ]:
# Calculate milestones
def get_time(act):
    return stg_df[stg_df["activity_name"] == act].groupby("case_id")["event_timestamp"].min()

reg_t = get_time("Registration")
triage_t = get_time("Triage")
xray_t = get_time("X-Ray")
doc_t = get_time("Doctor Consultation")
disp_t = stg_df.groupby("case_id")["event_timestamp"].max()

sla_df = fct_df[["case_id", "patient_gender", "patient_age_group", "clinical_severity", "primary_physician"]].copy()
sla_df["registration_time"] = sla_df["case_id"].map(reg_t)
sla_df["triage_time"] = sla_df["case_id"].map(triage_t)
sla_df["xray_time"] = sla_df["case_id"].map(xray_t)
sla_df["doc_time"] = sla_df["case_id"].map(doc_t)
sla_df["disp_time"] = sla_df["case_id"].map(disp_t)

entry = sla_df["registration_time"].combine_first(sla_df["triage_time"])
sla_df["door_to_doctor_min"] = ((sla_df["doc_time"] - entry).dt.total_seconds() / 60.0).round(1)
sla_df["triage_to_xray_min"] = ((sla_df["xray_time"] - sla_df["triage_time"]).dt.total_seconds() / 60.0).round(1)
sla_df["total_ed_los_min"] = ((sla_df["disp_time"] - entry).dt.total_seconds() / 60.0).round(1)

sla_df["door_to_doc_breach"] = sla_df["door_to_doctor_min"] > 30.0
sla_df["xray_breach"] = sla_df["triage_to_xray_min"] > 45.0
sla_df["ed_boarding_breach"] = sla_df["total_ed_los_min"] > 240.0

print(f"Door-to-Doctor Breach Rate: {sla_df['door_to_doc_breach'].mean()*100:.1f}%")
print(f"Radiology SLA Breach Rate:  {sla_df['xray_breach'].mean()*100:.1f}%")
print(f"ED Boarding Breach Rate:    {sla_df['ed_boarding_breach'].mean()*100:.1f}%")


In [ ]:
# 6. Parquet & CSV Export Verification
export_dict = {
    "stg_ehr_events": stg_df,
    "fct_patient_journeys": fct_df,
    "mart_bottlenecks": mart_bottlenecks,
    "mart_resource_handovers": mart_handovers,
    "mart_sla_compliance": sla_df
}

for name, df_obj in export_dict.items():
    csv_p = os.path.join(OUTPUT_DIR, f"{name}.csv")
    pq_p = os.path.join(OUTPUT_DIR, f"{name}.parquet")
    df_obj.to_csv(csv_p, index=False)
    df_obj.to_parquet(pq_p, index=False)
    print(f"[OK] Materialized {name}: {len(df_obj):,} rows -> {os.path.basename(csv_p)}, {os.path.basename(pq_p)}")
